In [1]:
import sys
import subprocess
import site
import os

print(f"Aktif Python Çekirdeği: {sys.executable}")
print("DergiPark için gerekli kütüphaneler kuruluyor ve yollar tanıtılıyor...")

try:
    # Gerekli paketleri User dizinine kur
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--user', 'sickle', 'lxml', 'beautifulsoup4', 'requests', 'pandas', 'tqdm'])
    
    # Kullanıcı dizinini sisteme enjekte et
    user_site = site.getusersitepackages()
    if user_site not in sys.path:
        sys.path.append(user_site)
        
    import importlib
    importlib.invalidate_caches()
    
    # İçe Aktarma Testi
    import requests
    from bs4 import BeautifulSoup
    import pandas as pd
    from tqdm.notebook import tqdm
    
    print("\n✅ TEST BAŞARILI: DergiPark modülleri zorla tanıtıldı ve hazır!")
    
except Exception as e:
    print(f"\n❌ HATA OLUŞTU: {e}")

Aktif Python Çekirdeği: c:\ProgramData\anaconda3\python.exe
DergiPark için gerekli kütüphaneler kuruluyor ve yollar tanıtılıyor...

✅ TEST BAŞARILI: DergiPark modülleri zorla tanıtıldı ve hazır!


In [2]:
import requests
import time
import pandas as pd
import os
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm

OUTPUT_FILE = "data/02_dergipark_abstracts.csv"
TARGET_LIMIT = 50000 # Çekilecek makale sayısını buradan artırabilirsin

# Eğer dosya yoksa başlıkları oluşturarak aç
if not os.path.isfile(OUTPUT_FILE):
    pd.DataFrame(columns=["source", "title", "abstract", "subjects"]).to_csv(OUTPUT_FILE, index=False, encoding="utf-8")

base_url = "https://dergipark.org.tr/api/public/oai"
params = {'verb': 'ListRecords', 'metadataPrefix': 'oai_dc'}

# Türkçe Tıp/Sağlık filtreleme kelimeleri (Projeye göre genişletilebilir)
medical_keywords = [
    'tıp', 'sağlık', 'hastane', 'klinik', 'tedavi', 'hasta', 
    'cerrahi', 'sendrom', 'enfeksiyon', 'hekim', 'hemşire', 
    'farmakoloji', 'psikiyatri', 'pediatri', 'kardiyoloji', 'diyabet'
]

dataset = []
count = 0

print(f"DergiPark OAI-PMH veri hattı başlatılıyor. Hedef: {TARGET_LIMIT} Abstract")

with tqdm(total=TARGET_LIMIT, desc="DergiPark Verileri") as pbar:
    while count < TARGET_LIMIT:
        try:
            response = requests.get(base_url, params=params, timeout=30)
            
            # Sunucu Bizi Yavaşlattığında (Anti-Ban Koruması)
            if response.status_code == 429:
                time.sleep(60)
                continue
                
            elif response.status_code != 200:
                time.sleep(15)
                continue

            # Gelen XML verisini parse et
            soup = BeautifulSoup(response.content, 'xml')
            
            error = soup.find('error')
            if error:
                print(f"\nOAI-PMH Hatası (Sonuna gelinmiş olabilir): {error.text}")
                break

            records = soup.find_all('record')
            
            for record in records:
                header = record.find('header')
                if header and header.get('status') == 'deleted':
                    continue
                    
                metadata = record.find('metadata')
                if metadata:
                    title_elem = metadata.find('dc:title')
                    desc_elem = metadata.find('dc:description')
                    subject_elems = metadata.find_all('dc:subject')
                    
                    if title_elem and desc_elem:
                        title = title_elem.text.strip()
                        description = desc_elem.text.strip()
                        subjects = ", ".join([sub.text for sub in subject_elems])
                        
                        text_to_search = (title + " " + description + " " + subjects).lower()
                        
                        # Filtreden Geçiyorsa
                        if any(word in text_to_search for word in medical_keywords):
                            dataset.append({
                                'source': 'DergiPark',
                                'title': title,
                                'abstract': description,
                                'subjects': subjects
                            })
                            count += 1
                            pbar.update(1)
                            
                            # Veri Kaybını Önlemek İçin 100 Adette Bir Diske Yaz
                            if len(dataset) >= 100:
                                pd.DataFrame(dataset).to_csv(OUTPUT_FILE, mode='a', header=False, index=False, encoding="utf-8")
                                dataset = []
                                
                            if count >= TARGET_LIMIT:
                                break
            
            # Sonraki sayfaya geçmek için jetonu (Token) al
            token = soup.find('resumptionToken')
            if token and token.text:
                params = {'verb': 'ListRecords', 'resumptionToken': token.text}
                time.sleep(3) # Kibar bot kuralı (Ban yememek için zorunlu)
            else:
                print("\nTüm veri tabanı tarandı, başka sayfa kalmadı.")
                break

        except requests.exceptions.RequestException as e:
            time.sleep(30)
            continue
        except Exception as e:
            print(f"\nBeklenmeyen hata: {e}")
            break

    # Kalan son partiyi kaydet
    if dataset:
        pd.DataFrame(dataset).to_csv(OUTPUT_FILE, mode='a', header=False, index=False, encoding="utf-8")

print(f"\nDergiPark işlemi tamamlandı! Veriler '{OUTPUT_FILE}' dosyasına kaydedildi.")

DergiPark OAI-PMH veri hattı başlatılıyor. Hedef: 50000 Abstract


DergiPark Verileri:   0%|          | 0/50000 [00:00<?, ?it/s]


Tüm veri tabanı tarandı, başka sayfa kalmadı.

DergiPark işlemi tamamlandı! Veriler 'data/02_dergipark_abstracts.csv' dosyasına kaydedildi.
